<a href="https://colab.research.google.com/github/anshubansal371/Multimodal-AI-Interview-Analyzer/blob/main/score_feedback.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers torch anthropic --quiet
!pip install openai-whisper --quiet
!apt-get install -y ffmpeg --quiet

import os, json, numpy as np
import torch
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')

DATA        = '/content/drive/MyDrive/Data'
MODEL_SAVE  = '/content/drive/MyDrive/Models'
ARRAYS_PATH = '/content/drive/MyDrive/Arrays'

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'cpu')

print(f"✅ TF      : {tf.__version__}")
print(f"✅ PyTorch : {torch.__version__}")
print(f"✅ GPU     : {torch.cuda.is_available()}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.7/831.7 kB 44.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
✅ TF      : 2.20.0
✅ PyTorch : 2.10.0+cu128
✅ GPU     : True


In [ ]:
# ── Load all 3 trained models ─────────────────────────
from transformers import (AutoTokenizer,
    AutoModelForSequenceClassification)

# Face model
face_model = tf.keras.models.load_model(
    f'{MODEL_SAVE}/face_model_best.h5')
FACE_EMOTIONS = {
    0:'angry',1:'disgust',2:'fear',
    3:'happy',4:'neutral',5:'sad',6:'surprise'}
print(f"✅ Face model : {face_model.input_shape}")

# Audio model
audio_model = tf.keras.models.load_model(
    f'{MODEL_SAVE}/audio_model_best.keras')
with open(f'{ARRAYS_PATH}/audio/audio_meta.json') as f:
    audio_meta = json.load(f)
AUDIO_EMOTIONS = audio_meta['emotions']
print(f"✅ Audio model : {audio_model.input_shape}")

# Text model
ROBERTA_PATH = '/content/drive/MyDrive/final_roberta_model'
roberta_tok  = AutoTokenizer.from_pretrained(
    ROBERTA_PATH)
roberta_model = AutoModelForSequenceClassification\
    .from_pretrained(ROBERTA_PATH).to(device)
roberta_model.eval()
with open(f'{ROBERTA_PATH}/emotion_map.json') as f:
    emotion2id = json.load(f)
id2emotion = {v:k for k,v in emotion2id.items()}
print(f"✅ Text model  : {id2emotion}")

# Fusion model
fusion_model = tf.keras.models.load_model(
    f'{MODEL_SAVE}/fusion_model_best.keras')
print(f"✅ Fusion model: {fusion_model.input_shape}")

PERF_LABELS = {
    0:'Poor', 1:'Average',
    2:'Good', 3:'Excellent'}

✅ Face model : (None, 48, 48, 3)
✅ Audio model : (None, 64, 64, 3)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Text model  : {0: 'angry', 1: 'anxious', 2: 'positive', 3: 'surprised'}
✅ Fusion model: (None, 30)


In [ ]:
# ── Score Mapper ──────────────────────────────────────
def map_to_score(fusion_probs,
                 speech_quality=None):
    """
    Map fusion probabilities to
    4 interview dimension scores (0-100)
    """
    p_poor, p_avg, p_good, p_exc = \
        fusion_probs

    # Overall score 0-100
    overall = (
        p_poor * 15 +
        p_avg  * 45 +
        p_good * 72 +
        p_exc  * 92)

    # Confidence score
    confidence = min(100, (
        p_exc * 95 +
        p_good * 72 +
        p_avg  * 45 +
        p_poor * 15))

    # Clarity from speech quality
    if speech_quality:
        clarity = (
            speech_quality.get(
                'clarity_score', 50) * 0.6 +
            speech_quality.get(
                'fluency_score', 50) * 0.4)
    else:
        clarity = overall

    # Emotional intelligence
    eq_score = min(100, (
        p_exc  * 90 +
        p_good * 68 +
        p_avg  * 42 +
        p_poor * 18))

    # Answer quality
    if speech_quality:
        wc = speech_quality.get(
            'total_words', 50)
        length_bonus = min(20,
            wc/300*20)
        answer_q = min(100,
            overall * 0.8 + length_bonus)
    else:
        answer_q = overall * 0.85

    return {
        'overall'       : round(overall, 1),
        'confidence'    : round(confidence, 1),
        'clarity'       : round(clarity, 1),
        'emotional_iq'  : round(eq_score, 1),
        'answer_quality': round(answer_q, 1)
    }

# Test
test_probs = np.array([0.01, 0.02, 0.07, 0.90])
scores = map_to_score(test_probs)
print("✅ Score mapping test:")
for k,v in scores.items():
    bar = '█' * int(v/5)
    print(f"  {k:15s}: {bar} {v}/100")

✅ Score mapping test:
  overall        : █████████████████ 88.9/100
  confidence     : ██████████████████ 91.6/100
  clarity        : █████████████████ 88.9/100
  emotional_iq   : █████████████████ 86.8/100
  answer_quality : ███████████████ 75.6/100


In [ ]:
# ── AI Feedback Generator ─────────────────────────────
import anthropic

def generate_feedback(scores,
                      face_emotion,
                      audio_emotion,
                      text_emotion,
                      transcript=None,
                      job_title=None,
                      api_key=None):
    """
    Generate personalized feedback
    using Claude API
    """
    if not api_key:
        # Rule-based fallback
        return generate_rule_based_feedback(
            scores, face_emotion,
            audio_emotion, text_emotion)

    client = anthropic.Anthropic(
        api_key=api_key)

    prompt = f"""You are an expert interview coach.
Analyze this candidate's interview performance and provide feedback.

Performance Scores:
- Overall Score     : {scores['overall']}/100
- Confidence        : {scores['confidence']}/100
- Communication Clarity: {scores['clarity']}/100
- Emotional Intelligence: {scores['emotional_iq']}/100
- Answer Quality    : {scores['answer_quality']}/100

Detected Emotions:
- Facial expression : {face_emotion}
- Voice tone        : {audio_emotion}
- Answer sentiment  : {text_emotion}

{f'Job Role: {job_title}' if job_title else ''}
{f'Transcript snippet: {transcript[:300]}...' if transcript else ''}

Provide:
1. 3 specific STRENGTHS observed
2. 3 specific AREAS TO IMPROVE
3. 2 STAR-method answer suggestions
4. 3 actionable IMPROVEMENT TIPS (High/Medium/Low priority)

Keep feedback constructive, specific and encouraging.
Format with clear sections."""

    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        messages=[
            {"role": "user",
             "content": prompt}])

    return message.content[0].text

def generate_rule_based_feedback(
        scores, face_em,
        audio_em, text_em):
    """Fallback rule-based feedback"""
    overall = scores['overall']
    feedback = {}

    # Strengths
    strengths = []
    if scores['confidence'] > 70:
        strengths.append(
            "Strong confident body language "
            "and vocal presence")
    if scores['clarity'] > 70:
        strengths.append(
            "Clear and well-structured "
            "communication style")
    if scores['answer_quality'] > 70:
        strengths.append(
            "Good use of specific examples "
            "in answers")
    if text_em == 'positive':
        strengths.append(
            "Positive and enthusiastic tone "
            "throughout")
    if not strengths:
        strengths = [
            "Shows genuine interest",
            "Willing to engage with questions"]
    feedback['strengths'] = strengths[:3]

    # Areas to improve
    improvements = []
    if scores['confidence'] < 60:
        improvements.append(
            "Work on maintaining eye contact "
            "and upright posture")
    if scores['clarity'] < 60:
        improvements.append(
            "Reduce filler words (um, uh, like) "
            "and slow down speech pace")
    if scores['answer_quality'] < 60:
        improvements.append(
            "Use STAR method (Situation, Task, "
            "Action, Result) for structured answers")
    if audio_em in ['fearful','sad','angry']:
        improvements.append(
            "Work on vocal confidence — practice "
            "speaking in a calm, steady tone")
    if not improvements:
        improvements = [
            "Continue refining answer specificity",
            "Practice more mock interviews"]
    feedback['improvements'] = improvements[:3]

    # Tips
    if overall >= 80:
        tips = [
            "HIGH: Research company culture deeply",
            "MED: Prepare 3-5 specific project examples",
            "LOW: Practice salary negotiation responses"]
    elif overall >= 60:
        tips = [
            "HIGH: Practice STAR method daily",
            "HIGH: Record yourself and review",
            "MED: Expand technical vocabulary"]
    else:
        tips = [
            "HIGH: Daily mock interview practice",
            "HIGH: Work on reducing filler words",
            "HIGH: Build confidence with mirror practice"]
    feedback['tips'] = tips

    # Performance label
    if overall >= 80:
        feedback['label'] = 'Excellent'
        feedback['message'] = \
            "Outstanding performance! You are " \
            "well prepared for this interview."
    elif overall >= 60:
        feedback['label'] = 'Good'
        feedback['message'] = \
            "Good performance with room to grow. " \
            "Focus on the improvement areas."
    elif overall >= 40:
        feedback['label'] = 'Average'
        feedback['message'] = \
            "Average performance. Consistent " \
            "practice will help you improve."
    else:
        feedback['label'] = 'Needs Work'
        feedback['message'] = \
            "Keep practicing! Focus on the " \
            "high priority tips first."

    return feedback

# Test rule-based feedback
test_scores = {
    'overall': 72.5,
    'confidence': 68.0,
    'clarity': 80.0,
    'emotional_iq': 70.0,
    'answer_quality': 71.9}

feedback = generate_rule_based_feedback(
    test_scores, 'happy', 'neutral', 'positive')

print("✅ Feedback generation test:")
print(f"\n  Label   : {feedback['label']}")
print(f"  Message : {feedback['message']}")
print(f"\n  Strengths:")
for s in feedback['strengths']:
    print(f"    ✅ {s}")
print(f"\n  Improvements:")
for i in feedback['improvements']:
    print(f"    ⚠️  {i}")
print(f"\n  Tips:")
for t in feedback['tips']:
    print(f"    💡 {t}")

✅ Feedback generation test:

  Label   : Good
  Message : Good performance with room to grow. Focus on the improvement areas.

  Strengths:
    ✅ Clear and well-structured communication style
    ✅ Good use of specific examples in answers
    ✅ Positive and enthusiastic tone throughout

  Improvements:
    ⚠️  Continue refining answer specificity
    ⚠️  Practice more mock interviews

  Tips:
    💡 HIGH: Practice STAR method daily
    💡 HIGH: Record yourself and review
    💡 MED: Expand technical vocabulary


In [ ]:
# ── Complete Interview Analyzer ───────────────────────
import re
from PIL import Image
import cv2, librosa

FACE_TO_DIM = {
    'angry':2,'disgust':2,'fear':2,
    'happy':1,'neutral':4,'sad':2,'surprise':3}
AUDIO_TO_DIM = {
    'angry':2,'disgust':2,'fearful':2,
    'happy':1,'neutral':4,'sad':2}
TEXT_TO_DIM  = {
    'angry':2,'anxious':2,
    'positive':1,'surprised':3}

FILLER_WORDS = [
    'um','uh','like','so','you know',
    'basically','literally','actually',
    'i mean','right','okay so']

def analyze_speech_quality(text):
    if not text: return {}
    words = text.lower().split()
    total = len(words)
    if total == 0: return {}
    fillers = sum(
        len(re.findall(
            r'\b'+re.escape(f)+r'\b',
            text.lower()))
        for f in FILLER_WORDS)
    ttr = len(set(words)) / total
    sents = [s for s in
             re.split(r'[.!?]+',
                      text.strip())
             if len(s.strip()) > 0]
    aws  = total / max(len(sents), 1)
    return {
        'total_words'        : total,
        'filler_count'       : fillers,
        'filler_ratio'       : round(fillers/total,3),
        'vocabulary_richness': round(ttr, 3),
        'clarity_score'      : round(
            max(0, min(100,
                (1-fillers/total*3)*100)), 1),
        'fluency_score'      : round(
            min(100, ttr*70 +
                min(aws/20,1)*30), 1),
        'total_words'        : total}

def build_fusion_vector(fr, ar, tr, sq=None):
    fp = np.array(fr['probs']+[0]*(7-len(fr['probs'])))[:7]
    ap = np.array(ar['probs']+[0]*(6-len(ar['probs'])))[:6]
    tp = np.array(tr['probs']+[0]*(4-len(tr['probs'])))[:4]
    co = np.array([fr['confidence'],
                   ar['confidence'],
                   tr['confidence']])
    dv = np.zeros(5)
    dv[fr['dim']] += 0.33
    dv[ar['dim']] += 0.33
    dv[tr['dim']] += 0.34
    sf = np.array([
        sq.get('clarity_score',50)/100,
        sq.get('fluency_score',50)/100,
        1-sq.get('filler_ratio',0),
        sq.get('vocabulary_richness',0.5),
        min(sq.get('total_words',50)/200,1)
    ]) if sq else np.array([.5,.5,.8,.5,.5])
    return np.concatenate(
        [fp,ap,tp,co,dv,sf]).astype(np.float32)

def analyze_interview(
        video_path=None,
        audio_path=None,
        text=None,
        job_title=None,
        api_key=None):
    """
    Complete multimodal interview analysis
    Input  : video/audio/text
    Output : scores + feedback + emotions
    """
    print("🎯 Starting analysis...")
    result = {}

    # ── Step 1: Transcribe ─────────────────────
    if not text and (video_path or audio_path):
        import whisper
        wm = whisper.load_model("base")
        src = audio_path or video_path
        print("  Transcribing...")
        out  = wm.transcribe(
            src, language='en',
            fp16=torch.cuda.is_available())
        text = out['text'].strip()
    result['transcript'] = text or ''

    # ── Step 2: Speech quality ─────────────────
    sq = analyze_speech_quality(text or '')
    result['speech_quality'] = sq

    # ── Step 3: Text emotion ───────────────────
    print("  Analyzing text...")
    if text:
        inp = roberta_tok(
            text, return_tensors='pt',
            truncation=True, max_length=256,
            padding=True).to(device)
        with torch.no_grad():
            out   = roberta_model(**inp)
            probs = torch.softmax(
                out.logits,
                dim=1).cpu().numpy()[0]
        pred = np.argmax(probs)
        text_r = {
            'emotion'   : id2emotion[pred],
            'confidence': float(probs[pred]),
            'probs'     : probs.tolist(),
            'dim'       : TEXT_TO_DIM.get(
                id2emotion[pred], 4)}
    else:
        text_r = {
            'emotion':'positive',
            'confidence':0.5,
            'probs':[0.1,0.1,0.7,0.1],
            'dim':1}
    result['text_emotion'] = text_r

    # ── Step 4: Placeholder face/audio ────────
    # In production: extract from video frames
    face_r = {
        'emotion'   : 'neutral',
        'confidence': 0.6,
        'probs'     : [0.1,0.05,0.05,0.2,
                       0.5,0.05,0.05],
        'dim'       : 4}
    audio_r = {
        'emotion'   : 'neutral',
        'confidence': 0.6,
        'probs'     : [0.1,0.05,0.6,
                       0.1,0.1,0.05],
        'dim'       : 4}
    result['face_emotion']  = face_r
    result['audio_emotion'] = audio_r

    # ── Step 5: Fusion ─────────────────────────
    print("  Running fusion...")
    vec   = build_fusion_vector(
        face_r, audio_r, text_r, sq)
    probs = fusion_model.predict(
        vec.reshape(1,-1), verbose=0)[0]
    pred  = np.argmax(probs)

    # ── Step 6: Score mapping ──────────────────
    scores = map_to_score(probs, sq)
    result['scores'] = scores
    result['label']  = PERF_LABELS[pred]
    result['fusion_probs'] = {
        PERF_LABELS[i]: round(float(p),3)
        for i,p in enumerate(probs)}

    # ── Step 7: Feedback ───────────────────────
    print("  Generating feedback...")
    feedback = generate_feedback(
        scores,
        face_r['emotion'],
        audio_r['emotion'],
        text_r['emotion'],
        text, job_title, api_key)
    result['feedback'] = feedback

    print("✅ Analysis complete!")
    return result

print("✅ Complete pipeline ready!")

✅ Complete pipeline ready!


In [ ]:
# ── Test full pipeline ─────────────────────────────────
sample = """
I led a cross-functional team to deliver a
machine learning project on time. The situation
was challenging because we had tight deadlines.
I analyzed the requirements, coordinated with
stakeholders, and implemented an optimized
solution using Python and TensorFlow.
The result was a 35 percent improvement in
prediction accuracy and positive client feedback.
I am very passionate about solving complex
problems and delivering high quality results.
"""

print("="*55)
print("COMPLETE INTERVIEW ANALYSIS TEST")
print("="*55)

result = analyze_interview(
    text=sample,
    job_title="Machine Learning Engineer")

print(f"\n📊 SCORES:")
for k, v in result['scores'].items():
    bar = '█' * int(v/5)
    print(f"  {k:15s}: {bar} {v}/100")

print(f"\n🏆 PERFORMANCE: {result['label']}")

print(f"\n🎭 EMOTIONS:")
print(f"  Text  : {result['text_emotion']['emotion']}")
print(f"  Face  : {result['face_emotion']['emotion']}")
print(f"  Audio : {result['audio_emotion']['emotion']}")

fb = result['feedback']
if isinstance(fb, dict):
    print(f"\n💬 FEEDBACK:")
    print(f"  {fb.get('message','')}")
    print(f"\n  ✅ Strengths:")
    for s in fb.get('strengths',[]):
        print(f"     {s}")
    print(f"\n  ⚠️  Improve:")
    for i in fb.get('improvements',[]):
        print(f"     {i}")
    print(f"\n  💡 Tips:")
    for t in fb.get('tips',[]):
        print(f"     {t}")
else:
    print(f"\n💬 FEEDBACK:\n{fb}")

COMPLETE INTERVIEW ANALYSIS TEST
🎯 Starting analysis...
  Analyzing text...
  Running fusion...
  Generating feedback...
✅ Analysis complete!

📊 SCORES:
  overall        : ██████████████ 70.5/100
  confidence     : ██████████████ 70.5999984741211/100
  clarity        : ██████████████████ 91.5/100
  emotional_iq   : █████████████ 66.80000305175781/100
  answer_quality : ████████████ 60.70000076293945/100

🏆 PERFORMANCE: Good

🎭 EMOTIONS:
  Text  : positive
  Face  : neutral
  Audio : neutral

💬 FEEDBACK:
  Good performance with room to grow. Focus on the improvement areas.

  ✅ Strengths:
     Strong confident body language and vocal presence
     Clear and well-structured communication style
     Positive and enthusiastic tone throughout

  ⚠️  Improve:
     Continue refining answer specificity
     Practice more mock interviews

  💡 Tips:
     HIGH: Practice STAR method daily
     HIGH: Record yourself and review
     MED: Expand technical vocabulary


In [ ]:
# ── Save complete pipeline config ─────────────────────
import json, os

PIPELINE_DIR = f'{MODEL_SAVE}/full_pipeline'
os.makedirs(PIPELINE_DIR, exist_ok=True)

pipeline_config = {
    'version'     : '1.0',
    'face_model'  : 'face_model_best.h5',
    'audio_model' : 'audio_model_best.keras',
    'text_model'  : '../final_roberta_model',
    'fusion_model': 'fusion_model_best.keras',
    'face_emotions' : FACE_EMOTIONS,
    'audio_emotions': AUDIO_EMOTIONS,
    'text_emotions' : id2emotion,
    'perf_labels'   : PERF_LABELS,
    'score_weights' : {
        'overall'       : 1.0,
        'confidence'    : 0.25,
        'clarity'       : 0.25,
        'emotional_iq'  : 0.25,
        'answer_quality': 0.25},
    'input_dim'     : 30,
    'num_classes'   : 4
}

with open(f'{PIPELINE_DIR}/config.json',
          'w') as f:
    json.dump(pipeline_config, f, indent=2)

print("✅ Pipeline config saved!")
print(f"   {PIPELINE_DIR}/config.json")

print("  Score mapping    : 0-100 ✅")
print("  Rule-based feedback: ✅")
print("  Claude API ready : ✅ (needs key)")
print("  Full pipeline    : tested ✅")
print("  Config saved     : ✅")

✅ Pipeline config saved!
   /content/drive/MyDrive/Models/full_pipeline/config.json
  Score mapping    : 0-100 ✅
  Rule-based feedback: ✅
  Claude API ready : ✅ (needs key)
  Full pipeline    : tested ✅
  Config saved     : ✅
